# Container Image Pre-Pulling

A practical refresher on **getting large container images onto (or into) Kubernetes nodes *before* the workload pod lands**, so GPU/ML pods start in seconds instead of minutes. Covers the AWS/EKS toolbox: **Bootstrap Scripts, Bottlerocket data-volume caching, Custom AMIs, DaemonSets, ECR Pull-Through Cache, Kubernetes Jobs, and the SOCI Snapshotter** for lazy loading.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction <a id="introduction"></a>

### What is it?

**Container image pre-pulling** is the practice of placing a container image — or just enough of it to start — onto a node **before** a workload pod is scheduled there, so that the slow `docker/containerd pull` step is removed from the pod-startup critical path. When a pod is scheduled, the kubelet must have the image in the node's containerd content store (`/var/lib/containerd`) before it can create the container. With `imagePullPolicy: IfNotPresent` (the default for tagged images), an image already on disk is reused instantly.

This matters acutely for **AI/ML** because ML images are *enormous*: a CUDA + cuDNN + PyTorch + NCCL base is often **6–15 GB**, and images that bake in model weights or vLLM/TensorRT-LLM can exceed **20–40 GB**. Pulling and decompressing that from Amazon ECR over the network can add **2–10 minutes** to every cold node — directly inflating autoscaling latency, inference cold-start time, and the queue time of burst training jobs.

### Why use it?

- **Faster scale-up**: when Karpenter or Cluster Autoscaler adds a GPU node, the image   is already present, so the pod goes `ContainerCreating -> Running` in seconds.
- **Cheaper GPU time**: a `p5`/`p4d`/`g5` node billed by the second should not sit idle   for minutes pulling an image while you pay for 8x H100/A100.
- **Resilience**: pre-staging via ECR Pull-Through Cache avoids Docker Hub rate limits   and internet egress on every node.
- **Predictable cold starts**: SOCI lazy loading lets a 20 GB inference image begin   serving before the whole image has transferred.

### When to use it?

- GPU clusters that **scale from zero** or scale up in bursts (spot reclamation, queue   spikes), where node-join latency dominates job start time.
- **Inference** services with tight cold-start SLOs that autoscale per request volume.
- Pipelines that launch many short-lived **Kubernetes Jobs** from the same heavy image.
- Any fleet repeatedly pulling the **same** few large images from a remote registry.

## Key Features <a id="key-features"></a>

### The pre-pulling toolbox

| Technique | What it does | Best for |
|-----------|--------------|----------|
| **Bootstrap Scripts** | EC2 user-data runs `ctr/crictl pull` at node boot, before the node is `Ready` | A small, known set of images; quick to adopt with Karpenter `userData` or MNG launch templates |
| **Custom AMIs** | Bake image layers into the AMI so they exist on disk at boot | Stable images that change rarely; lowest cold-start of all |
| **Bottlerocket Data Volume** | Pre-warm the `/var/lib/containerd` data volume from an EBS snapshot of cached layers | Karpenter + Bottlerocket fleets needing fast scale-up without rebuilding full AMIs |
| **DaemonSets** | A pod on every node pulls + pins the image, keeping kubelet's cache warm | Steady fleets where nodes live long enough to benefit |
| **Kubernetes Jobs** | A targeted Job warms specific nodes/node groups ahead of the real workload | Pre-staging before a known batch run |
| **ECR Pull-Through Cache** | ECR mirrors upstream registries into a regional private repo on first pull | Avoiding Docker Hub rate limits and internet egress; faster in-VPC pulls |
| **SOCI Snapshotter** | Seekable OCI index lets containerd lazily stream files on demand | Huge images where you cannot wait for a full pull (start before transfer completes) |

### Two strategies, not one

Everything above falls into **two families**:

1. **Eager placement** — physically put the full image on the node ahead of time    (bootstrap, AMI, Bottlerocket snapshot, DaemonSet, Job). Startup is instant but you    pay storage and must keep the cache fresh.
2. **Lazy loading** — don't pre-place anything; make the *pull* itself fast by    streaming only the bytes needed to start (SOCI / stargz). Startup is near-instant    without pre-staging, at the cost of slower first file access for cold layers.

## Architecture Overview <a id="architecture"></a>

A normal pull sits squarely on the pod-startup critical path:

```
Karpenter adds node -> kubelet Ready -> pod scheduled -> kubelet pulls image -> container starts
                                                         |__ 2-10 min for a 20 GB ML image __|
```

Pre-pulling removes (or shrinks) that middle step:

```
                          +----------------------------- EAGER PLACEMENT ----------------------------+
  Amazon ECR (registry) --+  bootstrap script  /  custom AMI  /  Bottlerocket EBS-snapshot cache     |
        |                 |  DaemonSet / pre-pull Job  -->  image already in /var/lib/containerd      |
        |                 +-----------------------------------------------------------------------------+
        |
        |                 +----------------------------- LAZY LOADING -------------------------------+
        +- SOCI index ----+  soci-snapshotter mounts a FUSE view; containerd starts the container     |
                          |  immediately and fetches file ranges on demand (HTTP range requests)      |
                          +-----------------------------------------------------------------------------+
```

### Components

1. **Registry (Amazon ECR)** — source of truth for image manifests and layers. With    **Pull-Through Cache**, ECR also mirrors upstream registries (Docker Hub, Quay,    GHCR, public ECR, registry.k8s.io) into a regional private repo.
2. **Node container runtime (containerd)** — stores pulled layers in the content store    under `/var/lib/containerd`. This is what we are trying to pre-populate.
3. **kubelet** — checks `imagePullPolicy`; with `IfNotPresent` it skips the pull when    the image digest is already present locally.
4. **Snapshotter plugin** — the containerd component that materializes layers into a    root filesystem. Default is `overlayfs` (needs the full layer); **soci-snapshotter**    replaces it for lazy, on-demand loading.
5. **Pre-warm mechanism** — user-data, AMI build pipeline, EBS snapshot, DaemonSet, or    Job — whichever gets bytes onto the node ahead of the workload.

## Installation <a id="installation"></a>

### Prerequisites

- An **EKS cluster** (or self-managed Kubernetes) using **containerd** as the runtime.
- Node provisioning you control: **Karpenter** (`EC2NodeClass`) or **managed node   groups** with a custom launch template.
- **Amazon ECR** access from nodes via the `AmazonEC2ContainerRegistryReadOnly` managed   policy on the node role (and ideally an **ECR VPC endpoint** + **S3 gateway   endpoint**, since ECR layers live in S3).
- For SOCI: containerd >= 1.6 and the **soci-snapshotter** plugin; the AWS CLI and the   `soci` binary to build indexes.

### Tooling you will use

The pre-pull techniques themselves are configuration, not a package install. The client tools are:

```bash
# Build / push SOCI indexes (lazy loading)
go install github.com/awslabs/soci-snapshotter/cmd/soci@latest

# Pull images on a node from a bootstrap script (containerd CLIs)
sudo ctr -n k8s.io images pull <registry>/<repo>:<tag>     # containerd native
sudo crictl pull <registry>/<repo>:<tag>                    # CRI-level (kubelet's view)

# Inspect what is already cached on a node
sudo crictl images
```

**Note**: the cell below only checks the Python environment for the small estimators used later in this notebook. It does not install any cluster components.

In [ ]:
# This notebook's runnable cells are pure-Python estimators and decision helpers --
# the real "installation" is cluster configuration (manifests / user-data) shown inline.
# No third-party packages are required; everything below uses the standard library.
import sys

print("Python", sys.version.split()[0])
print("Pre-pull helpers ready -- no external dependencies needed.")

## Basic Usage <a id="basic-usage"></a>

Below are the *minimal* viable forms of each eager technique. Pick one and grow from it.

### 1. Bootstrap script (Karpenter `EC2NodeClass` user-data)

For **Amazon Linux 2023 / AL2** nodes, append `ctr` pulls after the standard bootstrap:

```yaml
apiVersion: karpenter.k8s.aws/v1
kind: EC2NodeClass
metadata:
  name: gpu
spec:
  amiFamily: AL2023
  role: KarpenterNodeRole-mycluster
  userData: |
    #!/bin/bash
    # ... EKS bootstrap runs via the AL2023 nodeadm/MIME flow ...
    # Pre-pull the heavy training image into containerd before workloads land:
    for img in \
      123456789012.dkr.ecr.us-east-1.amazonaws.com/training:cuda12-pt2.4 \
      123456789012.dkr.ecr.us-east-1.amazonaws.com/vllm:0.6.0; do
      /usr/bin/ctr -n k8s.io images pull --user AWS:$(aws ecr get-login-password) "$img"
    done
```

### 2. Pre-pull DaemonSet (warm every node, keep it pinned)

```yaml
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: image-prepuller
  namespace: kube-system
spec:
  selector: { matchLabels: { app: image-prepuller } }
  template:
    metadata: { labels: { app: image-prepuller } }
    spec:
      # Run only on GPU nodes:
      nodeSelector: { "nvidia.com/gpu.present": "true" }
      tolerations: [{ key: "nvidia.com/gpu", operator: "Exists", effect: "NoSchedule" }]
      initContainers:
      # The init container pulls the heavy image; the pause container keeps it pinned.
      - name: prepull
        image: 123456789012.dkr.ecr.us-east-1.amazonaws.com/training:cuda12-pt2.4
        command: ["/bin/sh", "-c", "echo image cached on $(hostname) && exit 0"]
      containers:
      - name: pause
        image: registry.k8s.io/pause:3.9   # tiny; just keeps the pod alive
        resources: { requests: { cpu: 10m, memory: 16Mi } }
```

### 3. One-shot pre-pull Job (warm a node group before a batch run)

```yaml
apiVersion: batch/v1
kind: Job
metadata: { name: prepull-training-image }
spec:
  parallelism: 8            # one pod per node you want warmed
  completions: 8
  template:
    spec:
      affinity:
        podAntiAffinity:     # spread one pod per node
          requiredDuringSchedulingIgnoredDuringExecution:
          - labelSelector: { matchLabels: { job-name: prepull-training-image } }
            topologyKey: kubernetes.io/hostname
      restartPolicy: Never
      containers:
      - name: prepull
        image: 123456789012.dkr.ecr.us-east-1.amazonaws.com/training:cuda12-pt2.4
        command: ["true"]    # pulling the image is the whole point
```

The estimator below shows *why* this is worth it.

In [ ]:
# Estimate how much pod-startup latency pre-pulling saves.
# Cold pull time is dominated by: bytes transferred / effective bandwidth + decompress.

def pull_seconds(image_gb, bandwidth_gbps=2.0, decompress_gbps=3.0, overhead_s=8.0):
    """Approximate containerd pull+unpack time for a compressed image.

    image_gb        : on-disk (uncompressed) image size in GiB
    bandwidth_gbps  : effective registry->node throughput (Gbit/s); ECR-in-VPC ~2-5
    decompress_gbps : gzip/zstd decompress throughput (Gbit/s)
    overhead_s      : manifest fetch, auth, layer setup constant
    """
    # Images compress ~2x on the wire; transfer the compressed bytes, unpack the full size.
    compressed_gb = image_gb / 2.0
    transfer_s = (compressed_gb * 8) / bandwidth_gbps        # GiB*8 = Gbit
    unpack_s = (image_gb * 8) / decompress_gbps
    return overhead_s + transfer_s + unpack_s

scenarios = [
    ("CUDA+PyTorch base", 8),
    ("vLLM inference",    18),
    ("LLM w/ baked weights", 35),
]

print(f"{'image':<24}{'size':>6}{'cold pull':>12}{'pre-pulled':>12}{'SOCI start':>12}")
print("-" * 66)
for name, gb in scenarios:
    cold = pull_seconds(gb)
    prepulled = 2.0          # image already on disk: just container create
    soci = 6.0 + gb * 0.15   # lazy: start fast, small per-GB index/mount cost
    print(f"{name:<24}{gb:>4}GB{cold:>10.0f}s{prepulled:>11.0f}s{soci:>11.0f}s")

print("\nPre-pulling turns minutes of cold-pull latency into seconds of container-create.")

## Advanced Features <a id="advanced-features"></a>

### Bottlerocket data-volume cache (EBS snapshot)

**Bottlerocket** splits the OS volume from a **data volume** that holds `/var/lib/containerd` and `/var/lib/kubelet`. The trick: launch one node, pull your heavy images so they land in the data volume, snapshot that EBS volume, and then have **every future node boot from that snapshot** — so images are present at boot without rebuilding the OS AMI. Karpenter wires this via `blockDeviceMappings`:

```yaml
apiVersion: karpenter.k8s.aws/v1
kind: EC2NodeClass
metadata: { name: gpu-bottlerocket }
spec:
  amiFamily: Bottlerocket
  role: KarpenterNodeRole-mycluster
  blockDeviceMappings:
  - deviceName: /dev/xvda                 # Bottlerocket OS volume
    ebs: { volumeSize: 4Gi, volumeType: gp3 }
  - deviceName: /dev/xvdb                 # data volume = containerd + kubelet
    ebs:
      volumeSize: 200Gi
      volumeType: gp3
      snapshotID: snap-0cachedimages123   # <-- pre-warmed cache of image layers
      iops: 6000
      throughput: 500
```

Refresh the snapshot whenever the image set changes (automate with a CI job that boots a warmer node, pulls, and `aws ec2 create-snapshot`s the data volume).

### SOCI lazy loading (Seekable OCI)

SOCI builds a **separate index artifact** that maps files to byte ranges inside the existing image layers, so containerd can start the container and fetch only what the process actually reads:

```bash
# Build a SOCI index and push it alongside the image (no image rebuild needed):
soci create 123456789012.dkr.ecr.us-east-1.amazonaws.com/vllm:0.6.0
soci push  --user AWS:$(aws ecr get-login-password) \
           123456789012.dkr.ecr.us-east-1.amazonaws.com/vllm:0.6.0
```

On the node, configure containerd to use the snapshotter for SOCI-indexed images:

```toml
# /etc/containerd/config.toml
[proxy_plugins.soci]
  type = "snapshot"
  address = "/run/soci-snapshotter-grpc/soci-snapshotter-grpc.sock"

[plugins."io.containerd.grpc.v1.cri".containerd]
  snapshotter = "soci"
  disable_snapshot_annotations = false
```

AWS **Fargate** supports SOCI lazy loading out of the box for images that ship a SOCI index. The decision helper below picks a technique for you.

In [ ]:
# Recommend a pre-pull technique from a few constraints.

def recommend(image_churn, scaleup_seconds_budget, platform, image_gb):
    """Return (technique, rationale).

    image_churn           : 'low' | 'medium' | 'high' (how often the image tag changes)
    scaleup_seconds_budget: tolerable extra node-join latency in seconds
    platform              : 'fargate' | 'bottlerocket' | 'al2023' | 'self-managed'
    image_gb              : uncompressed image size in GiB
    """
    if platform == "fargate":
        return ("SOCI lazy loading",
                "Fargate has no node disk to pre-warm; ship a SOCI index so tasks "
                "stream layers on demand.")
    if image_gb >= 25 and scaleup_seconds_budget < 30:
        return ("SOCI lazy loading",
                "Image is too large to fully pull inside the latency budget; start "
                "the container before the transfer finishes.")
    if image_churn == "low":
        return ("Custom AMI (or Bottlerocket snapshot)",
                "Image rarely changes, so bake it in for the lowest possible cold start.")
    if platform == "bottlerocket":
        return ("Bottlerocket data-volume EBS-snapshot cache",
                "Refresh a data-volume snapshot in CI; new nodes boot with layers present.")
    if scaleup_seconds_budget < 60:
        return ("Bootstrap-script pull in userData",
                "Pull during node boot so the image is ready before the first pod schedules.")
    return ("Pre-pull DaemonSet + ECR Pull-Through Cache",
            "Keep a warm fleet cache and serve fast local pulls for new nodes.")

cases = [
    ("low",    45, "al2023",       8),
    ("high",   20, "fargate",     18),
    ("medium", 25, "bottlerocket",12),
    ("high",   15, "al2023",      35),
]
for churn, budget, plat, gb in cases:
    tech, why = recommend(churn, budget, plat, gb)
    print(f"churn={churn:<6} budget={budget:>2}s {plat:<13} {gb:>2}GB")
    print(f"   -> {tech}\n      {why}\n")

## Use Cases <a id="use-cases"></a>

### Use Case 1: Scale-from-zero GPU training

- **Context**: a research cluster keeps zero GPU nodes idle; Karpenter provisions   `p5.48xlarge` nodes on demand when a training Job is submitted. The 12 GB CUDA+PyTorch   image adds ~4 minutes of cold pull per node — multiplied across a 16-node job.
- **Implementation**: bake the base layers into a **Bottlerocket data-volume snapshot**   refreshed nightly; new nodes boot with the image present.
- **Results**: per-node `ContainerCreating` drops from ~4 min to <15 s; the whole job   starts training minutes sooner, saving expensive idle GPU-seconds.

### Use Case 2: Autoscaling LLM inference with cold-start SLO

- **Context**: a vLLM service scales replicas with request volume; the 20 GB image   (engine + weights) would blow a 30 s cold-start SLO if fully pulled.
- **Implementation**: build a **SOCI index** for the image; soci-snapshotter streams   layers so the server begins loading weights almost immediately.
- **Results**: new replicas reach `Running` in single-digit seconds; only the bytes the   process touches are fetched.

### Use Case 3: Bursty batch pipeline from one heavy image

- **Context**: an evaluation pipeline launches hundreds of short Jobs from the same   image; the first pod on each new node pays the full pull.
- **Implementation**: a **pre-pull Job** (one pod per node, pod anti-affinity) warms the   target node group; an **ECR Pull-Through Cache** keeps the upstream base local.
- **Results**: subsequent Jobs hit `IfNotPresent` and start instantly; no Docker Hub   rate-limit failures.

## Best Practices <a id="best-practices"></a>

1. **Keep `imagePullPolicy: IfNotPresent` and use immutable, digest-pinned tags.**    Pre-pulling only helps if the workload reuses the cached image rather than re-pulling    `:latest`. Pin to a digest or an immutable tag so the cached layers match.
2. **Match the pre-pull image *exactly* to the workload image (same digest).** A    different tag or platform variant means a cache miss and a full pull anyway.
3. **Automate cache freshness.** For AMIs and Bottlerocket snapshots, rebuild on every    image release in CI; a stale cache silently falls back to slow cold pulls.
4. **Front everything with ECR Pull-Through Cache + VPC endpoints.** This removes    internet egress, dodges Docker Hub rate limits, and keeps layer traffic on the S3    gateway endpoint.
5. **Split images into a stable base + thin app layer.** Bake/pre-pull the large, slow-    changing base (CUDA, framework) and let only the small app layer pull at runtime.
6. **Prefer lazy loading (SOCI) when images exceed your latency budget.** Past ~20-25 GB    it is often cheaper to stream than to pre-place.
7. **Pre-pull only on the node types that need it** (GPU nodeSelector/taints) so you do    not waste disk and pull bandwidth on general-purpose nodes.

## Common Pitfalls <a id="pitfalls"></a>

1. **`:latest` defeats the cache.** Untagged or `:latest` images default to    `imagePullPolicy: Always`, so kubelet re-pulls every time and your pre-pull was    wasted. Use a fixed tag/digest.
2. **DaemonSet/Job pre-pull still races scale-up.** On a brand-new node the pre-pull pod    *also* has to be scheduled and pull — it does not magically pre-exist. For just-in-    time scale-up, prefer bootstrap/AMI/snapshot which complete at boot.
3. **Stale AMI/snapshot caches.** Baking an image and forgetting to refresh it means    nodes hold an old layer set and the workload pulls the delta anyway (or worse, runs    stale code if tags are mutable).
4. **Disk exhaustion.** Caching many multi-GB images fills `/var/lib/containerd`; size    the data volume and prune unused images, or kubelet image GC will evict your warm cache.
5. **Forgetting SOCI indexes after a rebuild.** A new image push without a regenerated    SOCI index means soci-snapshotter falls back to a normal (full) pull.
6. **No ECR VPC endpoint.** Without it, every node pulls layers over the NAT gateway —    slow, costly, and rate-limit-prone.

## Performance Optimization <a id="performance"></a>

### Configuration tuning

- **EBS throughput for the cache volume**: on Bottlerocket/data-volume caches, the   bottleneck is often disk, not network. Use `gp3` with raised `iops`/`throughput` (e.g.   6000 IOPS / 500 MB/s) so unpacking large layers is not I/O-bound.
- **zstd-compressed layers**: build images with zstd (`--compression=zstd`) — it   decompresses faster than gzip, shrinking the unpack term of a cold pull.
- **Parallel layer pulls**: containerd's `max_concurrent_downloads` (default 3) can be   raised when the registry and network can sustain it.
- **ECR in-region + VPC endpoint**: keeps effective bandwidth high (2-5 Gbit/s) and   avoids NAT egress.
- **SOCI prefetch**: tune which layers are lazily loaded vs. eagerly fetched so latency-   critical files are not paged in on the hot path.

The helper below shows the I/O-vs-network trade-off when sizing a cache volume.

In [ ]:
# Where does cold-pull time go? Compare a network-bound vs disk-bound node.

def breakdown(image_gb, net_gbps, disk_mbps):
    compressed_gb = image_gb / 2.0
    transfer_s = (compressed_gb * 8) / net_gbps               # network term
    unpack_s = (image_gb * 1024) / disk_mbps                  # GiB->MiB / (MiB/s)
    return transfer_s, unpack_s

print(f"{'profile':<28}{'transfer':>10}{'unpack':>10}{'bottleneck':>14}")
print("-" * 62)
profiles = [
    ("low net, fast disk",   18, 1.0, 500),
    ("fast net, slow disk",  18, 5.0, 120),
    ("balanced gp3 + VPC",   18, 3.0, 300),
]
for name, gb, net, disk in profiles:
    t, u = breakdown(gb, net, disk)
    bottleneck = "network" if t > u else "disk I/O"
    print(f"{name:<28}{t:>8.0f}s{u:>8.0f}s{bottleneck:>14}")

print("\nRaise EBS iops/throughput when unpack dominates; add a VPC endpoint when transfer does.")

## Production Deployment <a id="deployment"></a>

Pre-pulling is **cluster/node configuration**, so "deployment" means wiring it into your node provisioner and CI.

### Managed node group launch template (bootstrap pull)

```bash
#!/bin/bash
# user-data for an AL2023 managed node group launch template
# (runs alongside the EKS nodeadm bootstrap)
REGION=us-east-1
ACCT=123456789012
aws ecr get-login-password --region $REGION \
  | ctr -n k8s.io images pull --user AWS:- \
      $ACCT.dkr.ecr.$REGION.amazonaws.com/training:cuda12-pt2.4
```

### ECR Pull-Through Cache rule (CLI)

```bash
aws ecr create-pull-through-cache-rule \
  --ecr-repository-prefix dockerhub \
  --upstream-registry-url registry-1.docker.io \
  --credential-arn arn:aws:secretsmanager:us-east-1:123456789012:secret:ecr-pullthrough-dockerhub
# Workloads then reference:  <acct>.dkr.ecr.<region>.amazonaws.com/dockerhub/library/python:3.11
```

### CI to refresh a Bottlerocket data-volume snapshot

```bash
# 1. Launch a warmer instance from the Bottlerocket AMI with a fresh data volume.
# 2. Pull every production image into containerd on it.
# 3. Snapshot the data volume and update the EC2NodeClass snapshotID.
NEW_SNAP=$(aws ec2 create-snapshot --volume-id $DATA_VOL \
  --description "image-cache $(date +%F)" --query SnapshotId --output text)
kubectl patch ec2nodeclass gpu-bottlerocket --type merge \
  -p "{\"spec\":{\"blockDeviceMappings\":[{\"deviceName\":\"/dev/xvdb\",\"ebs\":{\"snapshotID\":\"$NEW_SNAP\"}}]}}"
```

## Monitoring and Observability <a id="monitoring"></a>

### Key metrics to track

- **Image pull duration** — kubelet exposes `kubelet_image_pull_duration_seconds` (and   events show `Pulling`/`Pulled` with elapsed time). A near-zero histogram means your   cache is hitting; a fat tail means cold pulls are leaking through.
- **Pod startup latency** — `kubelet_pod_start_duration_seconds` / time from   `scheduled` to `Running`; this is the number pre-pulling is meant to shrink.
- **Cache hit rate** — fraction of pods reporting `Container image already present on   machine` vs. `Pulling image`.
- **Node disk usage** — `/var/lib/containerd` free space; watch for kubelet image GC   (`imageGCHighThresholdPercent`) evicting warm images.
- **ECR pull-through metrics / 429s** — registry throttling or Docker Hub rate-limit   errors indicate the cache is being bypassed.

### Logging best practices

- Surface kubelet `Pulled` events (they include image size and duration) into your log   pipeline so you can alert on regressions.
- Tag warmer/CI snapshot jobs with the image digest they cached, so a stale cache is   obvious from the snapshot description.
- For SOCI, watch `soci-snapshotter` logs for fallback-to-full-pull warnings (missing or   mismatched index).

## Troubleshooting <a id="troubleshooting"></a>

### Issue 1: Pods still show long `Pulling image` events despite pre-pulling

**Symptoms**: `kubectl describe pod` shows a multi-minute `Pulling` -> `Pulled` gap on new nodes.

**Cause**: digest mismatch (pre-pulled a different tag/arch), `imagePullPolicy: Always` from a `:latest` tag, or a stale cache that lacks the current layers.

**Solution**: pin an immutable tag/digest, set `IfNotPresent`, and verify the cached digest with `crictl images --digests` matches the workload's.

### Issue 2: Nodes run out of disk / warm images get evicted

**Symptoms**: `DiskPressure` taints, evicted pods, or images you pre-pulled are gone.

**Cause**: too many large images for the volume size; kubelet image GC reclaiming space above `imageGCHighThresholdPercent` (default 85%).

**Solution**: enlarge `/var/lib/containerd`, prune unused images, or lower the cached image set; consider lazy loading instead of caching everything.

### Issue 3: SOCI falls back to a full pull

**Symptoms**: a SOCI-indexed image still takes full-pull time; soci-snapshotter logs a fallback warning.

**Cause**: the image was rebuilt/pushed without regenerating and pushing a SOCI index, or the index's minimum-layer-size filter skipped the big layers.

**Solution**: regenerate the index after every push (`soci create && soci push`) in CI; check `soci create --min-layer-size`.

### Issue 4: Pull-through cache returns auth/connectivity errors

**Symptoms**: `denied` or timeouts pulling `.../dockerhub/...` images.

**Cause**: missing upstream credentials in Secrets Manager, missing ECR/S3 VPC endpoints, or node role lacking `ecr:BatchImportUpstreamImage`.

**Solution**: attach the upstream credential ARN, add the VPC endpoints, and grant the pull-through ECR permissions to the node role.

## Comparison with Alternatives <a id="comparison"></a>

### How the techniques compare

| Technique | Cold start | Setup cost | Cache freshness burden | Just-in-time scale-up |
|-----------|-----------|-----------|------------------------|-----------------------|
| **Custom AMI** | Lowest (on disk at boot) | High (AMI build pipeline) | High (rebuild per release) | Yes |
| **Bottlerocket data-volume snapshot** | Very low | Medium (CI snapshot job) | Medium (re-snapshot per release) | Yes |
| **Bootstrap script pull** | Low (pull at boot) | Low | None (pulls current tag) | Yes |
| **DaemonSet pre-pull** | Low *after* warm | Low | None | **No** (races new nodes) |
| **Kubernetes Job pre-pull** | Low *after* warm | Low | None | Partial (must run ahead) |
| **ECR Pull-Through Cache** | Medium (still pulls) | Low | None | N/A (speeds pulls) |
| **SOCI lazy loading** | Very low (stream) | Medium (build indexes) | Medium (re-index per push) | Yes |

### When to choose which

- **Lowest possible cold start, stable image** -> Custom AMI or Bottlerocket snapshot.
- **Karpenter scale-from-zero, image changes weekly** -> Bottlerocket snapshot or   bootstrap pull.
- **Huge images / tight SLO / Fargate** -> SOCI lazy loading.
- **Many short Jobs on a steady fleet** -> DaemonSet/Job warm cache + Pull-Through Cache.
- **Avoiding Docker Hub limits and egress** -> ECR Pull-Through Cache (combine with any   of the above).

## Resources <a id="resources"></a>

### Official documentation

- Amazon ECR Pull-Through Cache — https://docs.aws.amazon.com/AmazonECR/latest/userguide/pull-through-cache.html
- SOCI snapshotter (awslabs) — https://github.com/awslabs/soci-snapshotter
- AWS Fargate + SOCI lazy loading — https://docs.aws.amazon.com/AmazonECS/latest/developerguide/container-considerations.html
- Bottlerocket OS (data volume) — https://github.com/bottlerocket-os/bottlerocket
- Karpenter `EC2NodeClass` (userData / blockDeviceMappings) — https://karpenter.sh/docs/concepts/nodeclasses/
- containerd image pull / snapshotters — https://github.com/containerd/containerd/blob/main/docs/snapshotters/

### Guides and references

- AWS blog: "Reduce container startup time on Amazon EKS with SOCI" — https://aws.amazon.com/blogs/containers/
- EKS Best Practices Guide — scalability & node bootstrapping — https://docs.aws.amazon.com/eks/latest/best-practices/
- kubelet `imagePullPolicy` semantics — https://kubernetes.io/docs/concepts/containers/images/#image-pull-policy
- stargz-snapshotter (the upstream lazy-loading project SOCI builds on) — https://github.com/containerd/stargz-snapshotter

### Related technologies

- **Karpenter / Cluster Autoscaler** — the node provisioners whose latency pre-pulling targets.
- **EC2 Image Builder / Packer** — pipelines for baking custom AMIs.
- **Spegel** — peer-to-peer in-cluster image mirror (nodes share layers without re-pulling from the registry).